In [5]:
import os
from google.colab import userdata

try:
  os.environ["GROQ_API_KEY"]=userdata.get("GROQAPIKEY")
except Exception:
  print("Please add GROQAPIKEY to your colab secrets.")



In [6]:
from openai import OpenAI

client = OpenAI(
    api_key=os.environ["GROQ_API_KEY"],
    base_url="https://api.groq.com/openai/v1"
)

In [7]:
import json, functools
from openai import RateLimitError, APIConnectionError

MODEL = "openai/gpt-oss-20b"

BANK = {
    "savings account": "Deposit money and earn interest",
    "current account": "For frequent business transactions",
    "fixed deposit": "Deposit money for a fixed period",
    "upi": "Instant money transfer",
    "loans": "Borrow money from the bank",
    "debit card": "For payments and cash withdrawals"
}

SYSTEM = f"""ROLE: You are a friendly assistant for ABC Bank.
FACTS: Banking services: {json.dumps(BANK)}
RULES: Never ask for PIN, CVV, OTP or passwords.
       Never invent account details or interest rates.
       If unknown, say "Please contact bank support."
       Remember the conversation.
FORMAT: Under 50 words, polite, English or Telugu matching the customer."""

def log_call(func):
    @functools.wraps(func)
    def wrapper(*args, **kwargs):
        result = func(*args, **kwargs)
        print(f"[log] {func.__name__}{args} -> {result!r}")
        return result
    return wrapper

@log_call
def get_info(item: str) -> str:
    return BANK.get(item.lower().strip(), "Information not available.")

history = [{"role": "system", "content": SYSTEM}]
MAX_TURNS = 8

def chat(question: str) -> str:
    history.append({"role": "user", "content": question})
    recent = [history[0]] + history[1:][-MAX_TURNS:]
    try:
        response = client.chat.completions.create(
            model=MODEL, messages=recent, temperature=0.3
        )
        reply = response.choices[0].message.content
    except RateLimitError:
        reply = "Please try again in a minute."
    except APIConnectionError:
        reply = "Server connection failed. Try again."
    history.append({"role": "assistant", "content": reply})
    return reply

print("UPI:", get_info("upi"))

while True:
    text = input("You: ")
    if text.lower().strip() in ("bye", "exit", "quit"):
        print("Bot: Thank you for contacting ABC Bank!")
        break
    print("Bot:", chat(text))

[log] get_info('upi',) -> 'Instant money transfer'
UPI: Instant money transfer
You: what  is your main purpose
Bot: I’m here to help you with ABC Bank services—answering questions, guiding you through accounts, loans, UPI, and more.
You: what is the  maximum amount that I can  transform at a single transaction through savings account
Bot: I’m sorry, but I don’t have that detail. Please contact bank support.
You: ok bye
Bot: Goodbye! If you need anything else, feel free to reach out.
You: bye
Bot: Thank you for contacting ABC Bank!
